# SOC-Triage-Gym v3 — GRPO Training

**Hackathon: OpenEnv Apr 2026 — Theme #1 Multi-Agent Interactions + Fleet AI Oversight**

This notebook trains a Tier-1 SOC Analyst LLM on multi-agent team episodes using **GRPO** (Group Relative Policy Optimization).

### Environment Architecture
```
Alert Queue → [Tier-1 Analyst] → ESCALATE_TO_TIER2 → [Tier-2 Responder] → [SOC Manager]
               (40 steps)           (ticket bus)        (20 steps)          (8 steps)
                                                                    ↑
                                                         [Red-Team Generator]
                                                         (adaptive curriculum)
```

### Reward Structure
- **Per step**: `0.6 × role_specific + 0.4 × team_F1`
- **Episode final**: `0.6 × individual + 0.4 × team_F1` (prevents GRPO advantage collapse)

### Training Plan
| Phase | Role | Frozen Roles | GPU Time |
|-------|------|-------------|----------|
| Day 1 AM | Tier-1 | Scripted T2+Manager | ~4h H100 |
| Day 1 PM | Tier-2 | Trained T1 | ~3h H100 |
| Day 2 AM | Manager | Trained T1+T2 | ~2h H100 |
| Day 2 PM | Joint | All co-trained | ~4h H100 |

In [ ]:
# Mount Google Drive so trained weights and plots persist across sessions.
from google.colab import drive
drive.mount('/content/drive')


## 1. Install Dependencies

> **Before running cell 1:** Runtime → Change runtime type → **GPU** (T4 or better)
>
> **After running cells 1 + 2:** Runtime → **Restart session** → then continue from cell 3.
>
> unsloth patches the Python environment at install time and must be restarted before it can be imported.

In [ ]:
# ── Step 1: unsloth (Colab-compatible install from source) ──────────────────
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# ── Step 2: TRL + training stack ─────────────────────────────────────────────
# NOTES:
# - No mergekit / llm_blender / weave: we stub them in cell 3 (pydantic pin conflict).
# - pydantic>=2.11: TRL 0.24 uses pydantic models with torch.Tensor fields.
# - transformers<5: transformers 5.0 removed compat attrs (warnings_issued) that
#   PEFT/TRL still rely on. Pin until the stack catches up.
!pip install trl peft accelerate datasets "pydantic>=2.11" "transformers<5" -q

# ── Step 3: environment deps ─────────────────────────────────────────────────
!pip install fastapi uvicorn httpx matplotlib numpy -q

# ── Step 4: clone repo if not already present ───────────────────────────────
import os
if not os.path.exists('server') and not os.path.exists('openenv-2'):
    !git clone https://github.com/ROHITCRAFTSYT/-Metas-OpenEnv-2.git openenv-2 -q
    os.chdir('openenv-2')
elif os.path.exists('openenv-2') and not os.path.exists('server'):
    os.chdir('openenv-2')

print('✓ Install complete. Working dir:', os.getcwd())


In [ ]:
# ── Verify installs (unsloth is checked later — it needs a runtime restart) ──
import importlib, sys, os

# After a Runtime Restart, Colab resets CWD to /content. Re-enter the repo
# so later cells can find server.app, train_grpo, scenarios, etc.
if os.path.exists('openenv-2') and not os.path.exists('server'):
    os.chdir('openenv-2')
print('Working dir:', os.getcwd())

# ── Install stubs for optional TRL deps ──────────────────────────────────────
# TRL 0.24+ does lazy `from X... import ...` inside grpo_trainer for features
# we don't use (model-merging, LLM-as-judge blender, W&B Weave telemetry).
# Installing them pins pydantic or pulls enormous deps. This meta_path finder
# returns a MagicMock for any import starting with a stubbed prefix — TRL's
# import succeeds, we never touch the functionality.
import importlib.abc, importlib.machinery
from unittest.mock import MagicMock

_STUBBED_MODULES = ('mergekit', 'llm_blender', 'weave')

class _OptionalTRLStubLoader(importlib.abc.Loader):
    def create_module(self, spec):
        m = MagicMock()
        m.__name__ = spec.name
        m.__path__ = []
        m.__spec__ = spec
        return m
    def exec_module(self, module):
        pass

class _OptionalTRLStubFinder(importlib.abc.MetaPathFinder):
    def find_spec(self, fullname, path, target=None):
        for stub in _STUBBED_MODULES:
            if fullname == stub or fullname.startswith(stub + '.'):
                return importlib.machinery.ModuleSpec(fullname, _OptionalTRLStubLoader())
        return None

if not any(isinstance(f, _OptionalTRLStubFinder) for f in sys.meta_path):
    sys.meta_path.insert(0, _OptionalTRLStubFinder())
    print(f'✓ Stubs installed for: {", ".join(_STUBBED_MODULES)}')

checks = {
    'trl': 'trl',
    'peft': 'peft',
    'transformers': 'transformers',
    'fastapi': 'fastapi',
    'httpx': 'httpx',
    'pydantic': 'pydantic',
}
all_ok = True
for label, pkg in checks.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', getattr(mod, 'VERSION', 'unknown'))
        print(f'  ✓ {label} {ver}')
    except ImportError:
        print(f'  ✗ {label} MISSING — rerun the install cell', file=sys.stderr)
        all_ok = False

# Sanity check: pydantic must be >=2.11 for TRL 0.24's grpo_trainer.
import pydantic
_pv = tuple(int(x) for x in str(pydantic.VERSION).split('.')[:2])
if _pv < (2, 11):
    raise RuntimeError(
        f'pydantic {pydantic.VERSION} is too old for TRL 0.24+. '
        f'Run: !pip install -U "pydantic>=2.11" — then Runtime → Restart session.'
    )

# transformers 5.0 removed `warnings_issued` that PEFT/TRL depend on.
import transformers
_tv = tuple(int(x) for x in str(transformers.__version__).split('.')[:2])
if _tv >= (5, 0):
    print(f'  ⚠ transformers {transformers.__version__} may break PEFT/TRL. '
          f'Run: !pip install "transformers<5" -q and restart.')

if not all_ok:
    raise RuntimeError('Some packages are missing. Re-run cell 2 and check for errors.')

print('\n✓ Base packages present.')
print('\n⚠️  If you have NOT yet restarted after the install cell:')
print('    Runtime → Restart session, then re-run THIS cell and continue.')
print('    (unsloth patches the Python environment and only takes effect after restart)')


## 2. Start SOC-Triage-Gym Server

In [ ]:
import subprocess, time, httpx, sys, os

if os.path.exists('openenv-2') and not os.path.exists('server'):
    os.chdir('openenv-2')
assert os.path.exists('server'), f'server/ not found in {os.getcwd()} — run the install cell first'

server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '7860'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

probe = httpx.Client(base_url='http://localhost:7860', timeout=5.0)

print('Waiting for server...', end='', flush=True)
ready = False
for attempt in range(30):
    time.sleep(2)
    if server_proc.poll() is not None:
        err = server_proc.stderr.read(3000).decode(errors='replace')
        raise RuntimeError(f'Server process exited early:\n{err}')
    try:
        resp = probe.get('/health')
        if resp.status_code == 200:
            print(f' ready after {(attempt+1)*2}s')
            print('Health:', resp.json())
            ready = True
            break
    except (httpx.ConnectError, httpx.ReadTimeout, httpx.ConnectTimeout):
        print('.', end='', flush=True)
probe.close()
if not ready:
    server_proc.terminate()
    try:
        _, err_bytes = server_proc.communicate(timeout=5)
    except subprocess.TimeoutExpired:
        server_proc.kill()
        _, err_bytes = server_proc.communicate()
    raise RuntimeError(f'Server never became healthy.\nstderr:\n{err_bytes.decode(errors="replace")[:3000]}')

client = httpx.Client(base_url='http://localhost:7860', timeout=180.0)


## 3. Verify Environment — Run Sample Episode

In [ ]:
import json, httpx

# 120s timeout: if the LLM judge is enabled and a call is slow, this gives it
# headroom. The judge itself has a 5s internal timeout, but episodes with many
# manager steps can still exceed 30s cumulatively.
client = httpx.Client(base_url='http://localhost:7860', timeout=120.0)

# Test 1: Legacy solo mode (backward compat)
obs = client.post('/reset', json={'task_id': 'phishing', 'seed': 42}).json()
print(f"Solo mode — alerts: {len(obs['alert_queue'])}, mode: {obs.get('episode_mode')}")

# Test 2: Team mode
obs = client.post('/reset', json={'task_id': 'team_phishing_escalation', 'seed': 42, 'mode': 'team'}).json()
print(f"Team mode — alerts: {len(obs['alert_queue'])}, role: {obs.get('current_role')}, phase: {obs.get('current_phase')}")

# Test 3: Red-team generated scenario
gen_resp = client.post('/generate_scenario', json={'seed': 42})
gen_data = gen_resp.json()
print(f"Generated scenario — task_id: {gen_data.get('task_id')}, difficulty: {gen_data.get('difficulty_floor')}")

print('\n✓ Environment verified')

## 4. Baseline: Oracle Heuristic Reward Curves

Run the scripted oracle agent to establish baseline scores before training.

In [ ]:
import sys
sys.path.insert(0, '.')
from train_grpo import run_episode, oracle_action, TIER1_TASKS, SEEDS

# Collect baseline scores on 10 seeds × 2 tasks = 20 episodes
baseline_scores = []
print('Running baseline oracle episodes...')
for task_id in TIER1_TASKS:
    for seed in SEEDS[:10]:
        score, traj = run_episode(client, task_id=task_id, seed=seed, role_to_train='tier1')
        score = max(0.0, float(score))  # clamp: negative cumulative = 0 task score
        baseline_scores.append({'task': task_id, 'seed': seed, 'score': score})
        print(f'  {task_id} seed={seed}: {score:.4f}')

baseline_avg = sum(s['score'] for s in baseline_scores) / len(baseline_scores)
print(f'\nOracle baseline avg: {baseline_avg:.4f}')
print('(Scripted heuristic; trained model should exceed this)')

### 4b. Random-vs-oracle baseline gap

Establishes the measurable headroom an RL-trained policy has to close. This is the artifact for the **20 %-weighted *Showing Improvement in Rewards*** judging criterion — emitted *before* training so it's available even if the GRPO cell is skipped on a free-tier GPU.

In [ ]:
from train_grpo import _compare_baselines

# ROLE is set later in the build-dataset cell; default to tier1 for this pre-training plot.
_role_for_plot = globals().get('ROLE', 'tier1')

_compare_baselines(client, TIER1_TASKS, _role_for_plot, n_seeds=8)

# Surface the artifact inline. The helper writes:
#   reward_comparison_baseline_<role>.png
#   reward_comparison_baseline_<role>.csv
from IPython.display import Image, display
import os, shutil

_png = f'reward_comparison_baseline_{_role_for_plot}.png'
_csv = f'reward_comparison_baseline_{_role_for_plot}.csv'
if os.path.exists(_png):
    display(Image(_png))
    if os.path.exists('/content/drive/MyDrive'):
        _dst_dir = '/content/drive/MyDrive/soc_triage_gym_results'
        os.makedirs(_dst_dir, exist_ok=True)
        for f in (_png, _csv):
            if os.path.exists(f):
                shutil.copy(f, _dst_dir)
        print(f'Mirrored gap artifacts to {_dst_dir}')


## 5. Load Model with Unsloth (4-bit LoRA)

In [ ]:
from unsloth import FastLanguageModel
import os

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
HF_TOKEN = os.getenv('HF_TOKEN', '')
if not HF_TOKEN:
    print('[info] HF_TOKEN not set — using ungated model.')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=768,
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN or None,
)

if not getattr(tokenizer, 'chat_template', None):
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "<|im_start|>{{ message['role'] }}\n{{ message['content'] }}<|im_end|>\n"
        "{% endfor %}"
        "{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}"
    )
    print('[info] Injected ChatML template onto tokenizer.')

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

model.print_trainable_parameters()


## 6. Build Training Dataset from Environment

In [ ]:
from datasets import Dataset
from train_grpo import build_step_dataset, ROLE_SYSTEM_PROMPTS
import httpx

try:
    client.close()
except Exception:
    pass
client = httpx.Client(base_url='http://localhost:7860', timeout=180.0)
assert client.get('/health').status_code == 200, 'Server not healthy — rerun server cell.'

ROLE = 'tier1'
TRAIN_SEEDS = list(range(42, 42 + 20))

print(f'Building per-step dataset for role={ROLE}...')
raw_data = build_step_dataset(client, TIER1_TASKS, TRAIN_SEEDS, ROLE)

import random
random.seed(42)
random.shuffle(raw_data)

split = max(1, len(raw_data) // 10)
train_data = raw_data[split:]
eval_data = raw_data[:split]

train_dataset = Dataset.from_list([
    {'prompt': d['prompt'],
     'task_id': d['task_id'],
     'seed': d['seed'],
     'step_index': d.get('step_index', 0)}
    for d in train_data
])
print(f'Train: {len(train_dataset)} | Eval: {len(eval_data)}')


## 7. Define Reward Function

In [ ]:
import json
from train_grpo import make_reward_fn

reward_fn = make_reward_fn(client, ROLE)

# Sanity check: per-step reward on a known good T1 action at step 0.
# Derive the real alert_id from a fresh reset so seed drift doesn't silently
# produce -0.05 against a stale hard-coded ALT-TEAM-001.
_probe = client.post('/reset', json={'task_id': 'team_phishing_escalation',
                                      'seed': 42, 'mode': 'team'}).json()
_alerts = _probe.get('alert_queue') or []
_sample_aid = _alerts[0]['alert_id'] if _alerts else 'ALT-000'
_sample_ip = (next((ip for a in _alerts
                    for ip in (a.get('indicators') or {}).get('ip', [])), None)
              or '203.0.113.1')

test_completions = [[
    {'role': 'assistant', 'content': json.dumps({
        'action_type': 'enrich_indicator',
        'indicator': _sample_ip,
        'indicator_type': 'ip',
        'query_alert_id': _sample_aid,
        'role': 'tier1'
    })}
]]
test_rewards = reward_fn(
    prompts=['test'],
    completions=test_completions,
    task_id=['team_phishing_escalation'],
    seed=[42],
    step_index=[0],
)
print(f'Per-step reward sanity check (alert={_sample_aid}, ip={_sample_ip}): {test_rewards}')


## 8. GRPO Training (group=8)

In [ ]:
from trl import GRPOConfig, GRPOTrainer
import trl, inspect, torch

print(f'TRL version: {trl.__version__}')

# Detect precision support. T4 (Turing) does NOT support bf16 — it needs fp16.
# A100/H100 (Ampere+) support bf16. TRL 0.24 defaults bf16=True, which crashes
# on T4, so we set it explicitly based on the actual GPU capability.
_supports_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"} | bf16={_supports_bf16}')

# Build config kwargs — newer TRL dropped `beta`, older TRL doesn't accept
# `bf16`/`fp16` on GRPOConfig at all. Filter to what this version accepts.
_grpo_params = set(inspect.signature(GRPOConfig.__init__).parameters)

_candidate_kwargs = dict(
    output_dir=f'./soc_grpo_{ROLE}',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=8,              # GRPO group size
    max_prompt_length=512,
    max_completion_length=256,
    learning_rate=5e-6,
    temperature=1.0,
    logging_steps=5,
    save_steps=50,
    report_to=['none'],
    seed=42,
    bf16=_supports_bf16,
    fp16=(not _supports_bf16) and torch.cuda.is_available(),
)

# KL coefficient: renamed across TRL versions (beta → kl_coef).
if 'beta' in _grpo_params:
    _candidate_kwargs['beta'] = 0.04
elif 'kl_coef' in _grpo_params:
    _candidate_kwargs['kl_coef'] = 0.04

# Drop anything the installed TRL doesn't accept, so we stay forward-compatible.
config_kwargs = {k: v for k, v in _candidate_kwargs.items() if k in _grpo_params}
_dropped = sorted(set(_candidate_kwargs) - set(config_kwargs))
if _dropped:
    print(f'[info] GRPOConfig ignored (not in this TRL version): {_dropped}')

grpo_config = GRPOConfig(**config_kwargs)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn],
    args=grpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print(f'Training {len(train_dataset)} examples × 3 epochs, GRPO group=8')
trainer.train()


## 9. Save Model

In [ ]:
output_path = f'./soc_grpo_{ROLE}'

# Save LoRA adapter FIRST, then merged 16-bit.
# Rationale: save_pretrained_merged(save_method='merged_16bit') triggers an
# in-place dequantise + merge that can disturb the adapter state for a
# subsequent lora-only save. Doing LoRA first guarantees we always have
# reloadable adapters even if the merge step OOMs on Colab T4.
if hasattr(model, 'save_pretrained_merged'):
    try:
        model.save_pretrained_merged(f'{output_path}-lora', tokenizer, save_method='lora')
        print(f'✓ LoRA-only adapters saved to {output_path}-lora')
    except Exception as e:
        print(f'[warn] LoRA save failed: {e}')

    try:
        model.save_pretrained_merged(output_path, tokenizer, save_method='merged_16bit')
        print(f'✓ Merged 16-bit saved to {output_path}')
    except Exception as e:
        print(f'[warn] merged_16bit save failed (likely OOM on T4): {e}')
        print('       LoRA adapters above are sufficient for evaluation.')
else:
    trainer.save_model(output_path)
    tokenizer.save_pretrained(output_path)
    print(f'✓ Model saved to {output_path}')

# Mirror to Google Drive so weights survive Colab disconnects.
import os, shutil
_drive_root = '/content/drive/MyDrive/soc_triage_gym_results'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(_drive_root, exist_ok=True)
    for src in (f'{output_path}-lora', output_path):
        if os.path.exists(src):
            dst = os.path.join(_drive_root, os.path.basename(src))
            try:
                if os.path.exists(dst):
                    shutil.rmtree(dst)
                shutil.copytree(src, dst)
                print(f'✓ Mirrored {src} → {dst}')
            except Exception as e:
                print(f'[warn] Drive mirror failed for {src}: {e}')

# Optional: push to HuggingFace Hub
# model.push_to_hub_merged(f'YOUR_HF_USERNAME/soc-triage-gym-{ROLE}',
#                         tokenizer, save_method='merged_16bit', token=HF_TOKEN)


## 10. Post-Training Evaluation — Reward Curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from unsloth import FastLanguageModel
from train_grpo import parse_action_from_text, format_obs_prompt

FastLanguageModel.for_inference(model)

def run_trained_episode(task_id, seed):
    """Run a team episode where the trained model handles the target role."""
    obs = client.post('/reset', json={'task_id': task_id, 'seed': seed, 'mode': 'team'}).json()
    step = 0
    while not obs.get('done', False) and step < 80:
        step += 1
        role = obs.get('current_role', 'tier1')
        if role == ROLE:
            prompt_text = format_obs_prompt(obs, role, step)
            messages = [
                {'role': 'system', 'content': ROLE_SYSTEM_PROMPTS[ROLE]},
                {'role': 'user', 'content': prompt_text},
            ]
            # apply_chat_template may return Tensor OR BatchEncoding depending
            # on tokenizer/transformers version — normalise to a 2D Tensor.
            enc = tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
            )
            input_ids = enc if torch.is_tensor(enc) else enc['input_ids']
            input_ids = input_ids.to(model.device)
            with torch.no_grad():
                outputs = model.generate(
                    input_ids=input_ids,
                    max_new_tokens=128,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                )
            text = tokenizer.decode(
                outputs[0][input_ids.shape[1]:], skip_special_tokens=True
            )
            action = parse_action_from_text(text, role)
            if action is None:  # parse_action_from_text never returns None today, belt-and-braces
                action = {'action_type': 'noop', 'role': role}
        else:
            action = oracle_action(obs)
        step_resp = client.post(
            '/step', content=json.dumps(action),
            headers={'Content-Type': 'application/json'},
        )
        if step_resp.status_code != 200:
            break
        obs = step_resp.json()
    return float(obs.get('task_score') or obs.get('cumulative_reward', 0.0))

# Evaluate on held-out seeds
EVAL_SEEDS = list(range(100, 115))  # seeds not seen during training
trained_scores = []
print('Evaluating trained model...')
for task_id in TIER1_TASKS:
    for seed in EVAL_SEEDS:
        score = run_trained_episode(task_id, seed)
        trained_scores.append(score)
        print(f'  {task_id} seed={seed}: {score:.4f}')

trained_avg = sum(trained_scores) / len(trained_scores) if trained_scores else 0.0
print(f'\nTrained model avg: {trained_avg:.4f}')
print(f'Oracle baseline avg: {baseline_avg:.4f}')
if baseline_avg > 0:
    print(f'Improvement: {trained_avg - baseline_avg:+.4f} '
          f'({(trained_avg/baseline_avg - 1)*100:+.1f}% vs oracle)')
else:
    print(f'Improvement: {trained_avg - baseline_avg:+.4f}')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('SOC-Triage-Gym v2 — GRPO Training Results', fontsize=14, fontweight='bold')

ax = axes[0]
ax.bar(['Oracle Baseline', f'GRPO Trained\n(role={ROLE})'],
       [baseline_avg, trained_avg],
       color=['#4C72B0', '#DD8452'], alpha=0.85, edgecolor='black')
ax.set_ylim(0, max(1.0, trained_avg * 1.2))
ax.set_ylabel('Episode Score')
ax.set_title('Before vs After Training')
ax.axhline(0.5, linestyle='--', color='gray', alpha=0.5, label='0.5 threshold')
for i, v in enumerate([baseline_avg, trained_avg]):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

ax2 = axes[1]
bl_vals = [max(0.0, s['score']) for s in baseline_scores[:len(trained_scores)]]
x = range(1, len(trained_scores) + 1)
ax2.plot(x, bl_vals, 'o-', alpha=0.5, label='Oracle baseline', color='#4C72B0')
ax2.plot(x, trained_scores, 's-', alpha=0.7, label=f'GRPO trained ({ROLE})', color='#DD8452')
w = min(5, len(trained_scores))
if len(trained_scores) >= w and w >= 2:
    sm = np.convolve(trained_scores, np.ones(w) / w, mode='valid')
    ax2.plot(range(w, len(trained_scores) + 1), sm, linewidth=2.5,
             color='#C44E52', label='Trained (smoothed)')
ax2.set_xlabel('Evaluation Episode')
ax2.set_ylabel('Score')
ax2.set_title('Per-Episode Scores (Held-Out Seeds)')
ax2.set_ylim(0, max(1.0, max(trained_scores) * 1.1) if trained_scores else 1.0)
ax2.legend()
plt.tight_layout()

# CRITICAL: save BEFORE show(). Some matplotlib backends close the figure on
# show(), which silently produces a blank PNG if we save after.
fig.savefig('soc_grpo_results.png', dpi=150, bbox_inches='tight')
print('Saved: soc_grpo_results.png')

drive_path = '/content/drive/MyDrive/soc_triage_gym_results'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(drive_path, exist_ok=True)
    fig.savefig(f'{drive_path}/soc_grpo_results.png', dpi=150, bbox_inches='tight')
    print(f'Also saved to Google Drive: {drive_path}/soc_grpo_results.png')

plt.show()


## 11. Red-Team Curriculum — Blue Win Rate Oscillation

This is the **Theme #4 Self-Improvement money shot**: blue-team win rate oscillates around 0.5 as the Red-Team Generator adapts difficulty.

In [ ]:
import json
import sys
sys.path.insert(0, '.')
from scenarios.red_team_generator import RedTeamGenerator
from models import RedTeamConfig
from train_grpo import oracle_action  # defensive: works even if earlier cells were skipped
import matplotlib.pyplot as plt

# Simulate curriculum: N rounds of difficulty adaptation
NUM_ROUNDS = 15
EPISODES_PER_ROUND = 5

generator = RedTeamGenerator(seed=42)
difficulty_history = []
blue_win_rate_history = []

print('Running Red-Team curriculum simulation...')
for round_num in range(NUM_ROUNDS):
    round_scores = []
    for seed_offset in range(EPISODES_PER_ROUND):
        # Generate scenario
        gen_resp = client.post('/generate_scenario', json={
            'difficulty_floor': generator.config.difficulty_floor,
            'noise_density': generator.config.noise_density,
            'seed': generator.seed + seed_offset,
        })
        if gen_resp.status_code != 200:
            round_scores.append(0.5)
            continue
        # Blue team plays (oracle heuristic)
        obs = client.post('/reset', json={
            'task_id': 'red_team_generated',
            'seed': generator.seed + seed_offset,
            'mode': 'team',
        }).json()
        step = 0
        while not obs.get('done', False) and step < 50:
            step += 1
            action = oracle_action(obs)
            resp = client.post('/step', content=json.dumps(action),
                               headers={'Content-Type': 'application/json'})
            if resp.status_code != 200:
                break
            obs = resp.json()
        score = obs.get('task_score') or obs.get('cumulative_reward', 0.0)
        round_scores.append(float(score))

    blue_win_rate = sum(1 for s in round_scores if s > 0.5) / max(1, len(round_scores))
    diff = generator.config.difficulty_floor

    difficulty_history.append(diff)
    blue_win_rate_history.append(blue_win_rate)

    print(f'  Round {round_num+1:2d} | difficulty={diff:.2f} | blue_win_rate={blue_win_rate:.2f} | '
          f'scores={[f"{s:.2f}" for s in round_scores]}')

    # Adapt difficulty for next round
    generator = generator.adapt_difficulty(blue_win_rate)

# Plot the oscillation
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle('SOC-Triage-Gym v2 — Red-Team Curriculum (Theme #4 Self-Improvement)',
             fontweight='bold')

rounds = range(1, NUM_ROUNDS + 1)
ax1.plot(rounds, blue_win_rate_history, 'o-', color='#2196F3', linewidth=2,
         label='Blue win rate')
ax1.axhline(0.5, linestyle='--', color='red', alpha=0.7, label='Target 0.5')
ax1.fill_between(rounds, 0.4, 0.6, alpha=0.1, color='green', label='Sweet spot [0.4, 0.6]')
ax1.set_ylabel('Blue Win Rate')
ax1.set_ylim(0, 1)
ax1.legend()
ax1.set_title('Blue-Team Win Rate (oscillates around 0.5 as Red-Team adapts)')

ax2.plot(rounds, difficulty_history, 's-', color='#F44336', linewidth=2,
         label='Difficulty floor')
ax2.set_xlabel('Round')
ax2.set_ylabel('Difficulty Floor')
ax2.set_ylim(0, 1)
ax2.legend()
ax2.set_title('Red-Team Difficulty Adaptation (escalates when blue wins too often)')

plt.tight_layout()
# Save BEFORE show() so notebooks that close figures on show() still get the PNG.
fig.savefig('redteam_curriculum.png', dpi=150, bbox_inches='tight')
print('Saved: redteam_curriculum.png')

import os
_drive = '/content/drive/MyDrive/soc_triage_gym_results'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(_drive, exist_ok=True)
    fig.savefig(f'{_drive}/redteam_curriculum.png', dpi=150, bbox_inches='tight')
    print(f'Also saved to Google Drive: {_drive}/redteam_curriculum.png')

plt.show()


## 12. Summary

### Results

| Metric | Value |
|--------|-------|
| Oracle baseline avg score | See `baseline_avg` printed above |
| GRPO trained model avg score | See `trained_avg` printed above |
| Improvement | `trained_avg - baseline_avg` |
| Tasks trained on | team_phishing_escalation, team_lateral_team |
| Training role | Tier-1 (frozen T2 + Manager as oracle) |
| GRPO group size | 8 |
| Training examples | ~800 per-step rows (20 seeds × 2 tasks × ~20 T1 steps each) |
| Red-Team curriculum rounds | 15 |

### What makes this different from other OpenEnv submissions

- **Per-step GRPO reward**: each training example is one `(obs, step_index)` pair. Reward = env's immediate `step_reward` after applying the model's action at that state — not a full-episode oracle score.
- **Team reward blend**: `0.6 × role + 0.4 × Δteam_F1` — agents are penalized for solo-optimizing at the expense of team outcome.
- **Fleet AI oversight**: SOC Manager role uses an LLM judge (heuristic fallback) to score natural-language explanations of the team's decisions.
- **Self-improving curriculum**: Red-Team Generator adapts difficulty round-by-round, tracking blue-team skill level rather than using a fixed schedule.
- **Reward integrity**: 6 exploit vectors audited and fixed; all regression-tested.

### Theme coverage
- **Theme #1 Multi-Agent Interactions**: T1 → T2 → Manager with ticket bus and phase state machine
- **Fleet AI Scalable Oversight**: Manager audits, flags, and explains team behavior — scored by LLM judge
- **Theme #4 Self-Improvement**: Red-Team Generator co-evolves; curriculum plot shows oscillation

### Output files
- `soc_grpo_results.png` — before/after GRPO reward comparison
- `redteam_curriculum.png` — Theme #4 self-improvement oscillation curve
- `./soc_grpo_tier1/` — trained Tier-1 LoRA adapter weights